# import

In [ ]:
from itertools import permutations
import csv
import random
import math

: 

In [ ]:

class Container:
    def __init__(self,length = 590,width = 235, height = 239):
        self.length = length
        self.width = width
        self.height = height
        self.placed_boxes = []
        self.free_spaces = [
            {"x":0,
             "y":0,
             "z":0,
             "L":length,
             "W":width,
             "H":height,
            }
        ]
        self.unplaced_boxes = []

        pass
    # returns all possible orientations of a box, example (30,40,20) =>  ( (30,40,20),(20,40,30) ect... )
    def get_orientations(self, box):

        l, w, h = box["l"], box["w"], box["h"]

        if box["fragile"]:
             return list(set([
            (l, w, h),
            (w, l, h)
            ]))

        return list(set(permutations((l, w, h))))

    #checks wether a given orientation of a box fits in a given space
    def fits(self,l,w,h,space):
         return l <= space["L"] and w <= space["W"] and h <= space["H"]


    #use the Deppest bottom left to chose the best placement for a box iterating throught 
    # the free spaces and all the possible orientations for the box
    #returns the choosen space and the choosen orientaton for the box if it can fit
    def find_best_placement_dblf(self, box):

        best_choice = None

        #  try all orientations
        for (l, w, h) in self.get_orientations(box):

            #  try all free spaces
            for space in self.free_spaces:

                #  check feasibility
                if self.fits(l, w, h, space):

                    # DBLF priority
                    score = (space["z"], space["x"], space["y"])

                    #  keep best candidate
                    if best_choice is None or score < best_choice["score"]:

                        best_choice = {
                            "space": space,
                            "dims": (l, w, h),
                            "score": score
                        }

        return best_choice

    
    #after finding where to place the box, we place it, then we remove the free space used by splitting it into smaller free spaces
    def place_box(self,box,choice):
        if choice is None:
            self.unplaced_boxes.append({
            "id": box["id"],
            "reason": "no valid placement found"
            })
            return False  # cannot place

        space = choice["space"]
        l, w, h = choice["dims"]

        x, y, z = space["x"], space["y"], space["z"]

        placed = {
        "id": box["id"],
        "name": box["name"],
        "weight": box["weight"],
        "x": x,
        "y": y,
        "z": z,
        "l": l,
        "w": w,
        "h": h
        }

        self.placed_boxes.append(placed)

    # remove used space
        self.free_spaces.remove(space)

    # split space
        new_spaces = self.split_space(space, l, w, h, x, y, z)

        self.free_spaces.extend(new_spaces)

        return True

    # we split the free space used to other to smaller free spaces
    def split_space(self, space, l, w, h, x, y, z):

        new_spaces = []

        # Right
        if space["L"] > l:
            new_spaces.append({
                "x": x + l,
                "y": y,
                "z": z,
                "L": space["L"] - l,
                "W": space["W"],
                "H": space["H"]
            })

        # Front
        if space["W"] > w:
            new_spaces.append({
                "x": x,
                "y": y + w,
                "z": z,
                "L": l,                 # only the box's footprint width, no overlap with Right
                "W": space["W"] - w,
                "H": space["H"]
            })

        # Top
        if space["H"] > h:
            new_spaces.append({
                "x": x,
                "y": y,
                "z": z + h,
                "L": l,                 # box footprint only, no overlap with Right or Front
                "W": w,
                "H": space["H"] - h
            })

        return new_spaces


    def evaluate_stability(self) -> dict:
    

        if not self.placed_boxes:
            return {"weight_score": 0.0, "support_score": 0.0, "stability_score": 0.0}

        # ── 1. WEIGHT DISTRIBUTION SCORE ────────────────────────────────────────
        # Idea: for every pair of boxes (A, B), if A is heavier than B,
        # A should sit lower (smaller z). We count how many pairs respect
        # this rule and divide by total pairs → fraction in [0, 1].

        pairs_correct = 0
        pairs_total   = 0

        for i, box_a in enumerate(self.placed_boxes):
            for box_b in self.placed_boxes[i + 1:]:

                w_a = box_a["weight"]
                w_b = box_b["weight"]

                # z is the bottom face of the box
                z_a = box_a["z"]
                z_b = box_b["z"]

                if w_a == w_b:
                    pairs_correct += 1      # equal weight: no violation
                elif w_a > w_b:
                    if z_a <= z_b:          # heavier A is at same level or below B ✓
                        pairs_correct += 1
                else:
                    if z_b <= z_a:          # heavier B is at same level or below A ✓
                        pairs_correct += 1

                pairs_total += 1

        weight_score = (pairs_correct / pairs_total * 100) if pairs_total > 0 else 100.0


        # ── 2. SUPPORT SCORE ────────────────────────────────────────────────────
        # A box is "supported" if it either:
        #   (a) sits on the floor  (z == 0), OR
        #   (b) has at least MIN_OVERLAP_RATIO of its base footprint
        #       covered by the top face of boxes directly below it.
        #
        # "Directly below" means: another box whose top face (z + h)
        # equals this box's bottom (z), and whose XY footprint overlaps.

        MIN_OVERLAP_RATIO = 0.30    # 30 % of base area must be supported

        supported_count = 0

        for box in self.placed_boxes:

            # boxes resting on the floor are always supported
            if box["z"] == 0:
                supported_count += 1
                continue

            box_x1, box_x2 = box["x"], box["x"] + box["l"]
            box_y1, box_y2 = box["y"], box["y"] + box["w"]
            box_base_area  = box["l"] * box["w"]

            supported_area = 0.0

            for other in self.placed_boxes:

                if other["id"] == box["id"]:
                    continue

                # other's top face must align with this box's bottom face
                other_top = other["z"] + other["h"]
                if other_top != box["z"]:
                    continue

                # compute XY overlap between footprints
                ox1, ox2 = other["x"], other["x"] + other["l"]
                oy1, oy2 = other["y"], other["y"] + other["w"]

                overlap_x = max(0, min(box_x2, ox2) - max(box_x1, ox1))
                overlap_y = max(0, min(box_y2, oy2) - max(box_y1, oy1))
                supported_area += overlap_x * overlap_y

            ratio = supported_area / box_base_area if box_base_area > 0 else 0

            if ratio >= MIN_OVERLAP_RATIO:
                supported_count += 1

        support_score = (supported_count / len(self.placed_boxes)) * 100


        # ── 3. COMBINED STABILITY SCORE ──────────────────────────────────────────
        # Weight distribution and support contribute equally by default.
        # You can tune these weights later when you build your fitness function.

        WEIGHT_IMPORTANCE  = 0.4
        SUPPORT_IMPORTANCE = 0.6    # support matters more: a floating box is dangerous

        stability_score = (
            weight_score  * WEIGHT_IMPORTANCE +
            support_score * SUPPORT_IMPORTANCE
        )

        return {
            "weight_score"   : round(weight_score,    2),
            "support_score"  : round(support_score,   2),
            "stability_score": round(stability_score, 2),
        } 




    # now this function will take a list of boxes with the order and try box by box to place it in the container, and returns the number of placed and unplaced boxes and show them 
    
    def pack_boxes(self, boxes): 
         self.placed_boxes = []
         self.unplaced_boxes = []
         self.free_spaces = [{
         "x": 0,
         "y": 0,
         "z": 0,
         "L": self.length,
         "W": self.width,
         "H": self.height
          }]

         for box in boxes:

            choice = self.find_best_placement_dblf(box)

            self.place_box(box, choice)

        # statistics
         placed_count = len(self.placed_boxes)
         unplaced_count = len(self.unplaced_boxes)
         packed_volume = 0

         for box in self.placed_boxes:
            packed_volume += (
            box["l"] *
            box["w"] *
            box["h"]
            )

    # container volume
         container_volume = (
         self.length *
         self.width *
         self.height
         )

         utilization = (
         packed_volume / container_volume
         ) * 100
         stability = self.evaluate_stability()
         

         return {
        "placed": len(self.placed_boxes),
        "placed_count": len(self.placed_boxes),
        "unplaced": len(self.unplaced_boxes),
        "unplaced_count": len(self.unplaced_boxes),
        "utilization": utilization,
        "stability": stability,
        "placed_boxes": self.placed_boxes,
        "unplaced_boxes": self.unplaced_boxes
         }
        
    

        

def load_random_boxes(csv_path, n=200, seed=None):

    boxes = []

    if seed is not None:
        random.seed(seed)

    # open csv
    with open(csv_path, newline='', encoding='utf-8') as file:

        reader = csv.DictReader(file)

        # convert all rows to list
        rows = list(reader)

    # choose n random rows
    selected_rows = random.sample(rows, min(n, len(rows)))

    # convert rows to box dictionaries
    for row in selected_rows:

        box = {
            "id": row["id"],
            "name": row["name"],
            "l": int(row["length"]),
            "w": int(row["width"]),
            "h": int(row["height"]),
            "weight": float(row["weight"]),
            "fragile": row["fragile"].lower() == "true"
        }

        boxes.append(box)

    return boxes

: 

In [ ]:
class SimulatedAnnealing:
    def __init__(self,container,intitial_boxes,T=1000,alpha=0.95):
        self.container = container
        self.current = intitial_boxes
        self.T = T
        self.alpha = alpha

    def get_neighbor(self, solution):
        neighbor = solution.copy()

        i, j = random.sample(range(len(neighbor)), 2)
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

        return neighbor
    
    def evaluate(self, solution):
       

        result = self.container.pack_boxes(solution)

        utilization = result["utilization"]
        stability = result["stability"]["stability_score"]

        w_util = 0.7
        w_stab = 0.3

        return w_util * utilization + w_stab * stability
    

    def accept(self, current_score, new_score,T):
    
        # if new solution is better → always accept
        if new_score > current_score:
            return True
        T = max(T, 1e-10)
        # if worse → accept with probability
        delta = current_score - new_score
        probability = math.exp(-delta / T)

        return random.random() < probability
    

    def run(self,iterations=1000):

        # ── 1. INITIALIZATION ─────────────────────────────
        current_solution = self.current
        current_score = self.evaluate(current_solution)

        best_solution = current_solution
        best_score = current_score

        T = self.T   # initial temperature

        # ── 2. MAIN LOOP ───────────────────────────────────
        for _ in range(iterations):

            # generate neighbor
            neighbor = self.get_neighbor(current_solution)
            neighbor_score = self.evaluate(neighbor)

            # decide accept/reject
            if self.accept(current_score, neighbor_score,T):

                current_solution = neighbor
                current_score = neighbor_score

                # update best solution
                if neighbor_score > best_score:
                    best_solution = neighbor
                    best_score = neighbor_score

            # cool down
            T *= self.alpha
            

        # ── 3. RETURN RESULT ───────────────────────────────
        return {
            "best_solution": best_solution,
            "best_score": best_score,
            "best_result": self.container.pack_boxes(best_solution)  # final state
        }


In [ ]:
if __name__ == "__main__":
    container = Container()
    boxes = load_random_boxes("../data/data.csv")
    SA = SimulatedAnnealing(container, boxes)
    sa_result = SA.run(500)
    result = sa_result["best_result"]

    print("========== SIMULATED ANNEALING ==========")
    print(f"Placed boxes   : {result['placed']}")
    print(f"Unplaced boxes : {result['unplaced']}")
    print(f"Score          : {sa_result['best_score']:.4f}")
    print()
    print(f"utilisation : {result['utilization']}")
    print(f"stabilty : {result['stability']}")

    print("Placed boxes:")
    for box in result["placed_boxes"]:
        print(box)

    print()

    print("Unplaced boxes:")
    for box in result["unplaced_boxes"]:
        print(box)


In [ ]:
class GreedyBFD:
    """Greedy Best-Fit Decreasing on the shared Container: volume-decreasing order, best-fit placement."""

    def __init__(self, container, initial_boxes):
        self.container = container
        self.boxes = initial_boxes

    def _find_best_fit_placement(self, box):
        best_choice = None
        c = self.container
        for (l, w, h) in c.get_orientations(box):
            for space in c.free_spaces:
                if not c.fits(l, w, h, space):
                    continue
                leftover = (space["L"] * space["W"] * space["H"]) - (l * w * h)
                candidate_key = (leftover, space["z"], space["x"], space["y"])
                if best_choice is None or candidate_key < best_choice["key"]:
                    best_choice = {
                        "space": space,
                        "dims": (l, w, h),
                        "key": candidate_key,
                    }
        return best_choice

    def run(self):
        c = self.container
        sorted_boxes = sorted(
            self.boxes, key=lambda b: b["l"] * b["w"] * b["h"], reverse=True
        )
        c.placed_boxes = []
        c.unplaced_boxes = []
        c.free_spaces = [
            {
                "x": 0,
                "y": 0,
                "z": 0,
                "L": c.length,
                "W": c.width,
                "H": c.height,
            }
        ]
        for box in sorted_boxes:
            c.place_box(box, self._find_best_fit_placement(box))
        packed_volume = sum(b["l"] * b["w"] * b["h"] for b in c.placed_boxes)
        cv = c.length * c.width * c.height
        utilization = (packed_volume / cv) * 100 if cv else 0.0
        stability = c.evaluate_stability()
        best_result = {
            "placed": len(c.placed_boxes),
            "unplaced": len(c.unplaced_boxes),
            "utilization": utilization,
            "stability": stability,
            "placed_boxes": c.placed_boxes,
            "unplaced_boxes": c.unplaced_boxes,
        }
        w_util, w_stab = 0.7, 0.3
        best_score = w_util * utilization + w_stab * stability["stability_score"]
        return {
            "best_solution": sorted_boxes,
            "best_score": best_score,
            "best_result": best_result,
        }


In [ ]:
if __name__ == "__main__":
    container = Container()
    boxes = load_random_boxes("../data/data.csv")

    bfd = GreedyBFD(container, boxes)
    bfd_result = bfd.run()
    r_bfd = bfd_result["best_result"]
    print("========== GREEDY BFD (same container) ==========")
    print(f"Placed boxes   : {r_bfd['placed']}")
    print(f"Unplaced boxes : {r_bfd['unplaced']}")
    print(f"Score          : {bfd_result['best_score']:.4f}")
    print(f"Utilization    : {r_bfd['utilization']}")
    print(f"Stability      : {r_bfd['stability']}")
    print()
    print("Placed boxes:")
    for box in result["placed_boxes"]:
        print(box)

    print()

    print("Unplaced boxes:")
    for box in result["unplaced_boxes"]:
        print(box)


In [ ]:

class GeneticAlgorithm:
    def __init__(
        self,
        container,
        boxes,
        max_evals=40_000,     # ← ONLY TUNE THIS — controls total runtime
        pop_size=None,
        generations=None,
        crossover_rate=0.85,
        mutation_rate=None,
        tournament_k=None,
        elitism_count=None,
        w_util=0.7,
        w_stab=0.3,
        stagnation_limit=None,
    ):
        self.container      = container
        self.boxes          = boxes
        self.crossover_rate = crossover_rate
        self.w_util         = w_util
        self.w_stab         = w_stab
        self.history        = []

        n = len(boxes)

        # ── AUTO-SCALE FROM n AND max_evals ──────────────────────────
        self.pop_size         = pop_size      or min(200, max(50, n))
        self.generations      = generations   or max(100, max_evals // self.pop_size)
        self.mutation_rate    = mutation_rate or round(min(0.30, 2 / math.sqrt(n)), 3)
        self.tournament_k     = tournament_k  or max(3, round(0.05 * self.pop_size))
        self.elitism_count    = elitism_count or max(2, round(0.03 * self.pop_size))
        self.stagnation_limit = stagnation_limit or max(30, round(5 / self.mutation_rate))

        print(f"── GA Parameters (n={n} boxes) ─────────────────────")
        print(f"  pop_size         = {self.pop_size}")
        print(f"  generations      = {self.generations}")
        print(f"  crossover_rate   = {self.crossover_rate}")
        print(f"  mutation_rate    = {self.mutation_rate}")
        print(f"  tournament_k     = {self.tournament_k}")
        print(f"  elitism_count    = {self.elitism_count}")
        print(f"  stagnation_limit = {self.stagnation_limit}")
        print(f"  total evals      = {self.pop_size * self.generations:,}")
        print("─────────────────────────────────────────────────────")

    def evaluate(self, chromosome):
        result = self.container.pack_boxes(chromosome)
        return (self.w_util * result["utilization"]
                + self.w_stab * result["stability"]["stability_score"])

    def _init_population(self):
        population = []
        for _ in range(self.pop_size):
            chrom = self.boxes[:]
            random.shuffle(chrom)
            population.append(chrom)
        return population

    def _tournament_select(self, population, fitnesses):
        def one():
            contestants = random.sample(range(len(population)), self.tournament_k)
            return population[max(contestants, key=lambda i: fitnesses[i])]
        return one(), one()

    def _ox_crossover(self, p1, p2):
        n    = len(p1)
        a, b = sorted(random.sample(range(n), 2))

        def build(main, other):
            seg_ids      = {box["id"] for box in main[a:b+1]}
            child        = [None] * n
            child[a:b+1] = main[a:b+1]
            remainder    = [box for box in other if box["id"] not in seg_ids]
            for pos, box in zip([i for i in range(n) if child[i] is None], remainder):
                child[pos] = box
            return child

        return build(p1, p2), build(p2, p1)

    def _swap_mutate(self, chromosome):
        mutant = chromosome[:]
        # double swap for stronger perturbation
        i, j = random.sample(range(len(mutant)), 2)
        mutant[i], mutant[j] = mutant[j], mutant[i]
        i, j = random.sample(range(len(mutant)), 2)
        mutant[i], mutant[j] = mutant[j], mutant[i]
        return mutant

    def _restart(self, population, fitnesses):
        sorted_idx = sorted(range(len(population)),
                            key=lambda i: fitnesses[i], reverse=True)
        survivors  = [population[i][:] for i in sorted_idx[:self.pop_size // 2]]
        fresh = []
        for _ in range(self.pop_size - len(survivors)):
            chrom = self.boxes[:]
            random.shuffle(chrom)
            fresh.append(chrom)
        return survivors + fresh

    def run(self):
        population = self._init_population()
        fitnesses  = [self.evaluate(c) for c in population]

        best_idx      = max(range(len(population)), key=lambda i: fitnesses[i])
        best_solution = population[best_idx][:]
        best_score    = fitnesses[best_idx]
        self.history.append(best_score)
        stagnation_counter = 0
        print(f"Gen 000 | best score: {best_score:.4f}")

        for gen in range(1, self.generations + 1):

            sorted_idx     = sorted(range(len(population)),
                                    key=lambda i: fitnesses[i], reverse=True)
            new_population = [population[i][:] for i in sorted_idx[:self.elitism_count]]

            while len(new_population) < self.pop_size:
                p1, p2 = self._tournament_select(population, fitnesses)
                if random.random() < self.crossover_rate:
                    c1, c2 = self._ox_crossover(p1, p2)
                else:
                    c1, c2 = p1[:], p2[:]
                if random.random() < self.mutation_rate:
                    c1 = self._swap_mutate(c1)
                if random.random() < self.mutation_rate:
                    c2 = self._swap_mutate(c2)
                new_population.append(c1)
                if len(new_population) < self.pop_size:
                    new_population.append(c2)

            population = new_population
            fitnesses  = [self.evaluate(c) for c in population]

            gen_best = max(range(len(population)), key=lambda i: fitnesses[i])
            if fitnesses[gen_best] > best_score:
                best_score         = fitnesses[gen_best]
                best_solution      = population[gen_best][:]
                stagnation_counter = 0
            else:
                stagnation_counter += 1

            if stagnation_counter >= self.stagnation_limit:
                population         = self._restart(population, fitnesses)
                fitnesses          = [self.evaluate(c) for c in population]
                stagnation_counter = 0
                print(f"Gen {gen:03d} | *** RESTART ***")

            self.history.append(best_score)
            if gen % 20 == 0:
                print(f"Gen {gen:03d} | best score: {best_score:.4f}")

        print(f"Gen {self.generations:03d} | FINAL best score: {best_score:.4f}")

        return {
            "best_solution": best_solution,
            "best_score":    best_score,
            "best_result":   self.container.pack_boxes(best_solution),
            "history":       self.history,
        }


In [ ]:
if __name__ == "__main__":
    container = Container()
    boxes     = load_random_boxes("../data/data.csv", n=200)  # ← change n freely

    ga = GeneticAlgorithm(
        container=container,
        boxes=boxes,
        max_evals=40_000,    # ← tune this for runtime: 40k ≈ 10 min for n=200
    )

    ga_result = ga.run()
    result    = ga_result["best_result"]

    print()
    print("========== GA PACKING RESULTS ==========")
    print(f"Best score     : {ga_result['best_score']:.4f}")
    print(f"Placed boxes   : {result['placed']}")
    print(f"Unplaced boxes : {result['unplaced']}")
    print()
    print(f"Utilisation    : {result['utilization']:.2f} %")
    print(f"Stability      : {result['stability']}")
    print()
    print("Placed boxes:")
    for box in result["placed_boxes"]:
        print(box)
    print()
    print("Unplaced boxes:")
    for box in result["unplaced_boxes"]:
        print(box)